In [486]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re


In [487]:
df = pd.read_csv('D:/Study/data_science/underpriced-listing-predictor/data/02.parsed/all_appliances_parsed.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFeatures list length distribution per category:")
df['n_features'] = df['features'].str.count("',") + 1
print(df.groupby('category')['n_features'].describe())
print("\n--- Sample rows ---")
for cat in df['category'].unique():
    print(f"\n=== {cat} ===")
    print(df[df['category']==cat].iloc[0][['name', 'price', 'rating', 'features']].to_dict())

Shape: (2925, 5)
Columns: ['name', 'price', 'rating', 'category', 'features']

Features list length distribution per category:
                 count       mean       std  min   25%   50%   75%   max
category                                                                
AC              1020.0  10.945098  1.139412  5.0  10.0  11.0  12.0  15.0
Refrigerator    1020.0   8.255882  1.433585  2.0   8.0   8.0   9.0  11.0
WashingMachine   885.0  11.027119  1.348432  4.0  10.0  11.0  12.0  18.0

--- Sample rows ---

=== AC ===
{'name': 'Whirlpool SAI18B52MCD1 1.5 Ton 5 Star Inverter Split AC', 'price': '₹24,990', 'rating': '--rating: 4.65;', 'features': "['Split AC', '1.5 Ton Capacity', 'Air Swing', 'Auto Restart', 'Turbo Mode', 'Sleep Mode', 'Dust Filter', 'Self Diagnosis', 'Night Glow Buttons', '5 Star Rating']"}

=== Refrigerator ===
{'name': 'Samsung RT28C3733S8 236 L 3 Star Double Door Refrigerator', 'price': '₹25,490', 'rating': '--rating:4.3;', 'features': "['Multi Door', 'Top Mounted F

In [488]:
# Let us check for null values
print(df.isnull().sum()) # Zero null values in our data

name          0
price         0
rating        0
category      0
features      0
n_features    0
dtype: int64


- No Null values found

### **UNIVERSAL COLUMNS**

### ***1. Cleaning Price Column***

In [489]:
df['price'] = pd.to_numeric(df['price'].str.replace('[₹,]', '', regex=True), errors='coerce')

print(df.info())
print(df['price'].sample(5))

# Check how many rows became NaN during coercion
nans = df['price'].isna().sum()
print(f"Price cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some prices were not converted. Inspecting first few NaNs:")
    print(df[df['price'].isna()].head())

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        2925 non-null   str  
 1   price       2925 non-null   int64
 2   rating      2925 non-null   str  
 3   category    2925 non-null   str  
 4   features    2925 non-null   str  
 5   n_features  2925 non-null   int64
dtypes: int64(2), str(4)
memory usage: 882.1 KB
None
856      37500
1339    138990
1457     28990
993      29999
2325     16990
Name: price, dtype: int64
Price cleaning: 0 rows coerced to NaN


### ***2. Cleaning Rating Column***

In [490]:
df['rating'] = pd.to_numeric(df['rating'].str.split(':').str[1].str.strip().str.replace(';','' , regex=True),errors='coerce')
print(df.info())
print(df['rating'].sample(5))

# Check how many rows became NaN during coercion
nans = df['rating'].isna().sum()
print(f"Rating cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some ratings were not converted. Inspecting first few NaNs:")
    print(df[df['rating'].isna()].head())

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        2925 non-null   str    
 1   price       2925 non-null   int64  
 2   rating      2925 non-null   float64
 3   category    2925 non-null   str    
 4   features    2925 non-null   str    
 5   n_features  2925 non-null   int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 841.0 KB
None
194     4.35
2592    4.75
668     4.40
2761    4.00
1362    4.15
Name: rating, dtype: float64
Rating cleaning: 0 rows coerced to NaN


### ***3. Cleaning Name Column***

**(i) Brand Name***

In [491]:
# 1. Extract brand name (first word)
df['brand_name'] = df['name'].str.split(" ").str[0]

# 2. Define the correction map
brand_corrections = {
    "O": "O General",
    "Blue": "Blue Star"
    # Only if they are different
    # Add the other broken brands you found here...
}

# 3. Apply the correction
df['brand_name'] = df['brand_name'].replace(brand_corrections)

# 4. Verify the results
print("Brand name distribution:")
print(df['brand_name'].value_counts().head(10))

Brand name distribution:
brand_name
Haier        315
Samsung      269
LG           264
Whirlpool    256
Godrej       234
Voltas       219
Blue Star    166
IFB          151
Lloyd        142
Panasonic    123
Name: count, dtype: int64


**(ii) Capacity Column**

In [492]:
# This assigns the two captured groups directly to new columns in your df
df[['capacity_value', 'capacity_unit']] = df['features'].str.extract(r'(\d+\.?\d*)\s*(Ton|-Ton|L|-L|Kg|-Kg)',re.IGNORECASE)

# Now convert the value column to float, because extract returns everything as strings
df['capacity_value'] = pd.to_numeric(df['capacity_value'], errors='coerce')

In [493]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2925 non-null   str    
 1   price           2925 non-null   int64  
 2   rating          2925 non-null   float64
 3   category        2925 non-null   str    
 4   features        2925 non-null   str    
 5   n_features      2925 non-null   int64  
 6   brand_name      2925 non-null   object 
 7   capacity_value  2923 non-null   float64
 8   capacity_unit   2923 non-null   str    
dtypes: float64(2), int64(2), object(1), str(4)
memory usage: 915.6+ KB


In [494]:
idx =df[(df['capacity_value'].isna())].index.to_list()

In [495]:
df.loc[idx, ['capacity_value', 'capacity_unit']] = [[2.02, 'Ton'],[30,'kg']]
df[(df['capacity_value'].isna())]


,name,price,rating,category,features,n_features,brand_name,capacity_value,capacity_unit


In [496]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1020
L      1020
kg      883
Kg        2
Name: count, dtype: int64

In [497]:
idx = df[df['capacity_unit']=='Kg'].index.to_list()
df.loc[idx,'capacity_unit']='kg'

In [498]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1020
L      1020
kg      885
Name: count, dtype: int64

***(iii) Extract Star Rating from Name***

In [499]:
# Extract star rating from name using regex - handles:
# - Digits followed by optional spaces, optional hyphen, optional spaces, then "star" (case-insensitive)
# Examples: "5 Star", "4-Star", "3-star", "2 star", etc.
df['star_rating_from_name'] = df['name'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)

# Convert to numeric (handle any non-numeric values)
df['star_rating_from_name'] = pd.to_numeric(df['star_rating_from_name'], errors='coerce')


**Finding Star Ratings From Features**

In [500]:
df['star_rating_from_features'] = df['features'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)
df['star_rating_from_features'] = pd.to_numeric(df['star_rating_from_features'], errors='coerce')

In [501]:
# 3. Fill NaNs in the main column using the features-extracted values
df['star_rating'] = df['star_rating_from_name'].fillna(df['star_rating_from_features'])

In [502]:
# Add binary indicator for categories that typically have energy star ratings
df['has_star_rating'] = df['category'].isin(['AC', 'Refrigerator']).astype(int)


In [503]:
idx = df[df['category']=='WashingMachine'].index.to_list()
df.loc[idx,'star_rating'] = np.nan
# filled all the star ratings of WashingMachine as nan 

In [504]:
# drop the temporary columns
del df['star_rating_from_features'] ,df['star_rating_from_name']

In [505]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             2925 non-null   str    
 1   price            2925 non-null   int64  
 2   rating           2925 non-null   float64
 3   category         2925 non-null   str    
 4   features         2925 non-null   str    
 5   n_features       2925 non-null   int64  
 6   brand_name       2925 non-null   object 
 7   capacity_value   2925 non-null   float64
 8   capacity_unit    2925 non-null   str    
 9   star_rating      1957 non-null   float64
 10  has_star_rating  2925 non-null   int64  
dtypes: float64(3), int64(3), object(1), str(4)
memory usage: 961.0+ KB


### **UNIVERSAL FEATURES**

In [506]:
# has_inverter
df['has_inverter'] = df['features'].str.contains(r'Inverter',case=False).astype(int)

In [507]:
# has_wifi, has_app_control, has_voice_control
df['has_wifi'] = df['features'].str.contains(r'Wi-fi',case=False).astype(int)
df['has_voice_control'] = df['features'].str.contains(r'Voice',case=False).astype(int)
df['has_app_control'] = df['features'].str.contains(r'APP',case=False).astype(int)


In [508]:
# smart_connectivity_score (0 to 3) Total
df['smart_connectivity_score'] = df['has_wifi']+df['has_voice_control']+df['has_app_control']

In [509]:
# model_year
df['model_year'] = df['name'].str.extract(r'\b(201[0-9]|202[0-9])\b').astype('Int64')

# is_recent_model
df['is_recent_model'] = np.where(
    df['model_year'].isna(),
    np.nan,
    (df['model_year'] >= 2024).astype('Int64')
)
df['is_recent_model'] = df['is_recent_model'].astype('Int64')

# appliance_age

df['appliance_age'] = df['model_year'].apply(lambda x: 2026 - x).astype('Int64')

### ***AC_Specific Columns***

In [510]:
# Split AC and Window AC binary column. Also remove ACs other than Split and Windows.

ac_df = df[df['category']=='AC']
df['ac_split'] = ac_df['features'].str.contains(r'Split' , re.IGNORECASE).astype('Int64')
df['ac_window'] = ac_df['features'].str.contains(r'Window' , re.IGNORECASE).astype('Int64')

idx = ac_df[~(ac_df['features'].str.contains(r'Split|Window'))].index.to_list() # indices of AC otherthan Split|window
df = df.drop(idx, errors='ignore')


In [511]:
# has pm25 filter
df['ac_pm25_filter'] = ac_df['features'].str.contains(
    r'pm\s*2\.?5',
    case=False,
    na=False,
    regex=True
).astype('Int64')

# has hepa filter
ac_df['ac_hepa_filter'] = ac_df['features'].str.contains(
    r'hepa',
    case=False,
    na=False,
    regex=True
).astype('Int64')

# has auto clean
ac_df['ac_auto_clean'] = ac_df['features'].str.contains(
    r'auto clean',
    case=False,
    na=False,
    regex=True
).astype('Int64')

# has Hot and Cold
ac_df['ac_hot_and_cold'] = ac_df['features'].str.contains(
    r'hot|cold',
    case=False,
    na=False,
    regex=True
).astype('Int64')

# has Copper condenser
ac_df['ac_hot_and_cold'] = ac_df['features'].str.contains(
    r'copper|condenser',
    case=False,
    na=False,
    regex=True
).astype('Int64')





In [ ]:
import ast

df['features'] = df['features'].apply(ast.literal_eval) # This converts dtype of features column into list

In [ ]:
# Apply the counter to check the 10 most common Features
from collections import Counter

all_features = []
ac_df = df[df['category']=='AC']

import ast

ac_df['features'] = ac_df['features'].apply(ast.literal_eval) # This converts dtype of features column into list

for row in ac_df['features']:
    all_features.extend(row)

best_features = Counter(all_features).most_common()
binary_feat = []
for i in best_features:
    binary_feat.append(i[0])

print(binary_feat)

['Dust Filter', 'Auto Restart', 'Sleep Mode', 'Split AC', 'Dehumidification', 'Inverter', 'Turbo Mode', '1.5 Ton Capacity', 'Air Swing', 'Self Diagnosis']


In [530]:
remove_features = [
    'Dust Filter', 'Auto Restart', 'Sleep Mode', 'Split AC',
    'Inverter','1.5 Ton Capacity', 'Air Swing'
]

binary_feat = [feat for feat in binary_feat if feat not in remove_features]

print(binary_feat)

['Dehumidification', 'Turbo Mode', 'Self Diagnosis']


In [531]:
for feat in binary_feat:
    df[f'ac_{feat}'] = ac_df['features'].apply(lambda x : int(feat in x))
    df[f'ac_{feat}'] = df[f'ac_{feat}'].astype('Int64')

print(df.info())


<class 'pandas.DataFrame'>
Index: 2917 entries, 0 to 2924
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2917 non-null   str    
 1   price                     2917 non-null   int64  
 2   rating                    2917 non-null   float64
 3   category                  2917 non-null   str    
 4   features                  2917 non-null   str    
 5   n_features                2917 non-null   int64  
 6   brand_name                2917 non-null   object 
 7   capacity_value            2917 non-null   float64
 8   capacity_unit             2917 non-null   str    
 9   star_rating               1949 non-null   float64
 10  has_star_rating           2917 non-null   int64  
 11  has_inverter              2917 non-null   int64  
 12  has_wifi                  2917 non-null   int64  
 13  has_voice_control         2917 non-null   int64  
 14  has_app_control         